# Personal finance cleaning notebook

This notebook turns the raw Excel workbook into a clean, Tableau-ready transaction CSV.

Raw source: `personal_finances_raw.xlsx`

Final output: `data/cleaned/personal_finance_transactions_clean.csv`

Important principle: do not manually clean the Excel workbook. Keep the raw file messy but trustworthy, and put every repeatable cleaning decision here or in `data/reference/classification_rules.csv`.

## 1. Setup

If imports fail, run this from the project folder:

```bash
uv venv .venv
uv pip install --python .venv/bin/python -r requirements.txt
```

Then use the `.venv` Python interpreter as the notebook kernel.

In [ ]:
from pathlib import Path
import re
from difflib import get_close_matches

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)
sns.set_theme(style="whitegrid")

In [ ]:
PROJECT_DIR = Path.cwd()
RAW_FILE = PROJECT_DIR / "personal_finances_raw.xlsx"
RULES_FILE = PROJECT_DIR / "data" / "reference" / "classification_rules.csv"
CLEAN_DIR = PROJECT_DIR / "data" / "cleaned"
QUALITY_DIR = PROJECT_DIR / "data" / "quality"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
QUALITY_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_FILE.exists():
    raise FileNotFoundError(f"Raw workbook not found: {RAW_FILE}")
if not RULES_FILE.exists():
    raise FileNotFoundError(f"Classification rules file not found: {RULES_FILE}")

RAW_FILE, RULES_FILE

## 2. Load and inspect the raw Excel file

The workbook is read with `header=None` because the first rows are not a clean table header. They contain labels, totals, and summary cells.

In [ ]:
raw = pd.read_excel(RAW_FILE, sheet_name="Sheet1", header=None, engine="openpyxl")
raw["source_row"] = raw.index + 1

print(f"Raw shape: {raw.shape}")
display(raw.head(15))

## 3. Keep the transaction area and rename columns

Inspection shows the real transactions begin around Excel row 6.

Useful source columns:

- A: date
- B/C: food amount/detail
- D/E: other things amount/detail
- F/G: subscriptions amount/detail
- H: housing/utilities amount
- I: transport amount
- K/L: family amount/detail

In [ ]:
FIRST_DATA_ROW = 6

column_map = {
    0: "date",
    1: "food",
    2: "food_detail",
    3: "other_things",
    4: "other_things_detail",
    5: "subscriptions",
    6: "subscriptions_detail",
    7: "housing_utilities",
    8: "transport",
    10: "family",
    11: "family_detail",
    "source_row": "source_row",
}

data = raw.loc[raw["source_row"] >= FIRST_DATA_ROW, list(column_map.keys())].rename(columns=column_map).copy()

display(data.head(12))

## 4. Clean and forward-fill dates

Continuation rows often have a blank date. Those rows belong to the most recent date above them, so the date is forward-filled.

In [ ]:
def parse_excel_dates(series: pd.Series) -> pd.Series:
    """Parse Excel serial dates and normal text dates safely."""
    numeric = pd.to_numeric(series, errors="coerce")
    result = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")

    excel_serial_mask = numeric.between(20000, 60000)
    result.loc[excel_serial_mask] = pd.to_datetime(
        numeric.loc[excel_serial_mask], unit="D", origin="1899-12-30", errors="coerce"
    )

    text_mask = ~excel_serial_mask
    result.loc[text_mask] = pd.to_datetime(series.loc[text_mask], errors="coerce")
    return result


data["date"] = parse_excel_dates(data["date"]).ffill()

missing_dates = data["date"].isna().sum()
print("Missing dates after forward fill:", missing_dates)
assert missing_dates == 0, "Some rows still have missing dates after forward-fill."

display(data[["source_row", "date", "food", "food_detail", "other_things", "other_things_detail"]].head(12))

## 5. Reshape from wide format to one transaction per row

The raw sheet allows several transactions on one Excel row. Tableau will work much better with a tidy table: one transaction per row.

In [ ]:
detailed_categories = {
    "food": "food_detail",
    "other_things": "other_things_detail",
    "subscriptions": "subscriptions_detail",
    "family": "family_detail",
}

simple_categories = ["housing_utilities", "transport"]

rows = []
for amount_col, detail_col in detailed_categories.items():
    temp = data[["date", amount_col, detail_col, "source_row"]].copy()
    temp = temp.rename(columns={amount_col: "amount", detail_col: "detail_raw"})
    temp["category"] = amount_col
    rows.append(temp)

for amount_col in simple_categories:
    temp = data[["date", amount_col, "source_row"]].copy()
    temp = temp.rename(columns={amount_col: "amount"})
    temp["detail_raw"] = pd.NA
    temp["category"] = amount_col
    rows.append(temp)

tidy = pd.concat(rows, ignore_index=True)
tidy["amount"] = pd.to_numeric(tidy["amount"], errors="coerce")
tidy = tidy.dropna(subset=["amount"]).copy()
tidy = tidy[tidy["amount"] > 0].copy()

tidy = tidy[["date", "category", "amount", "detail_raw", "source_row"]]
print(f"Tidy transactions: {len(tidy):,}")
display(tidy.head(20))

## 6. Normalize detail names

This step reduces clerical work. Instead of manually classifying `SM Supermarket`, `sm supermarket`, and `SM  Supermarket` separately, they become the same normalized value: `sm_supermarket`.

In [ ]:
def clean_detail(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().lower()
    value = re.sub(r"\s+", "_", value)
    value = re.sub(r"[^\wæøåÆØÅéÉüÜäÄöÖñÑ-]", "", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_") or pd.NA


tidy["detail_clean"] = tidy["detail_raw"].apply(clean_detail)

display(tidy[["detail_raw", "detail_clean"]].drop_duplicates().head(30))

## 7. Classification strategy

The old notebook put hundreds of classification decisions inside notebook cells. That works once, but it becomes tedious and brittle.

This notebook uses a better system:

1. Exact rules live in `data/reference/classification_rules.csv`.
2. Rules can be global or category-specific using `category_scope`.
3. Keyword rules handle obvious new variants automatically.
4. Unmapped names are exported to `data/quality/unmapped_details.csv` for review.

Next time you add expenses, you should mostly edit the CSV rule file, not rewrite notebook code.

In [ ]:
rules = pd.read_csv(RULES_FILE).fillna("")

expected_rule_cols = {
    "category_scope", "detail_clean", "detail_grouped", "shopping_type",
    "expense_type_override", "recurrence_override", "note"
}
missing_rule_cols = expected_rule_cols - set(rules.columns)
assert not missing_rule_cols, f"Missing columns in classification rules: {missing_rule_cols}"

rules = rules.drop_duplicates(subset=["category_scope", "detail_clean"], keep="last")
print(f"Loaded classification rules: {len(rules):,}")
display(rules.head(10))

In [ ]:
global_rules = rules[rules["category_scope"].eq("")].set_index("detail_clean")
scoped_rules = rules[~rules["category_scope"].eq("")].set_index(["category_scope", "detail_clean"])

keyword_rules = [
    (r"supermarket|market|grocery|groceries|vegetable|fruit|rice|egg|fish|pork|chicken|water|vand", "groceries"),
    (r"mcdonald|jollibee|kfc|chowking|restaurant|pizza|burger|ramen|shawarma|foodpanda", "restaurants"),
    (r"coffee|donut|bakery|dessert|icecream|sugar_drink|cocacola|alcohol", "snacks_drinks"),
    (r"shopee|lazada|amazon|unitop|mr_diy|miniso|clothes|slipper|shoe|book|gadget|phone|usb", "shopping"),
    (r"dentist|medical|clinic|dermatologi", "healthcare"),
    (r"gym|netflix|curiositystream|subscription", "subscriptions"),
    (r"load|sim|wifi|internet", "mobile_data"),
    (r"church|donation|gcash_til_ivy|family", "family"),
    (r"tricycle|tricicle|gasoline|benzin|bike|bus|taxi|grab", "transport"),
    (r"electricity|strom|water_bill|rent|lege|gas", "housing_utilities"),
]


def rule_lookup(row, column):
    detail = row["detail_clean"]
    if pd.isna(detail) or detail == "":
        return ""

    scoped_key = (row["category"], detail)
    if scoped_key in scoped_rules.index:
        value = scoped_rules.loc[scoped_key, column]
        if isinstance(value, pd.Series):
            value = value.iloc[-1]
        if value != "":
            return value

    if detail in global_rules.index:
        value = global_rules.loc[detail, column]
        if isinstance(value, pd.Series):
            value = value.iloc[-1]
        if value != "":
            return value

    return ""


def keyword_group(detail):
    if pd.isna(detail) or detail == "":
        return ""
    for pattern, group in keyword_rules:
        if re.search(pattern, detail):
            return group
    return ""


tidy["detail_grouped"] = tidy.apply(lambda row: rule_lookup(row, "detail_grouped"), axis=1)
tidy["classification_source"] = np.where(tidy["detail_grouped"].ne(""), "exact_rule", "")

keyword_matches = tidy["detail_grouped"].eq("") & tidy["detail_clean"].notna()
tidy.loc[keyword_matches, "detail_grouped"] = tidy.loc[keyword_matches, "detail_clean"].apply(keyword_group)
tidy.loc[keyword_matches & tidy["detail_grouped"].ne(""), "classification_source"] = "keyword_rule"

no_detail = tidy["detail_clean"].isna()
tidy.loc[no_detail, "detail_grouped"] = tidy.loc[no_detail, "category"]
tidy.loc[no_detail, "classification_source"] = "category_default_no_detail"

still_unmapped = tidy["detail_grouped"].eq("")
tidy.loc[still_unmapped, "detail_grouped"] = tidy.loc[still_unmapped, "category"]
tidy.loc[still_unmapped, "classification_source"] = "needs_review"

# Optional fields from rules.
tidy["shopping_type"] = tidy.apply(lambda row: rule_lookup(row, "shopping_type"), axis=1).replace("", pd.NA)
tidy["expense_type_override"] = tidy.apply(lambda row: rule_lookup(row, "expense_type_override"), axis=1)
tidy["recurrence_override"] = tidy.apply(lambda row: rule_lookup(row, "recurrence_override"), axis=1)

display(tidy.head(20))

## 8. Expense type and recurrence

Default assumptions:

- Most spending is `operating`.
- Most spending is `recurring`.
- Exact rules can override those defaults.
- Construction is treated separately.

In [ ]:
tidy["expense_type"] = "operating"
tidy.loc[tidy["expense_type_override"].eq("capital"), "expense_type"] = "capital"
tidy.loc[tidy["detail_grouped"].eq("construction"), "expense_type"] = "construction"

tidy["recurrence"] = "recurring"
tidy.loc[tidy["recurrence_override"].eq("one_off"), "recurrence"] = "one_off"
tidy.loc[tidy["detail_grouped"].eq("construction"), "recurrence"] = "construction"

tidy = tidy.drop(columns=["expense_type_override", "recurrence_override"])

display(tidy.groupby(["expense_type", "recurrence"], dropna=False)["amount"].sum().reset_index())

## 9. Add Tableau-friendly date columns and source metadata

In [ ]:
tidy["year"] = tidy["date"].dt.year
tidy["month"] = tidy["date"].dt.month
tidy["month_name"] = tidy["date"].dt.month_name()
tidy["year_month"] = tidy["date"].dt.to_period("M").astype(str)
tidy["source_file"] = RAW_FILE.name
tidy["source_sheet"] = "Sheet1"

final_columns = [
    "date", "year", "month", "month_name", "year_month",
    "category", "amount",
    "detail_raw", "detail_clean", "detail_grouped", "classification_source",
    "shopping_type", "expense_type", "recurrence",
    "source_file", "source_sheet", "source_row",
]

tidy = tidy[final_columns].sort_values(["date", "source_row", "category"]).reset_index(drop=True)

display(tidy.head(30))

## 10. Data quality checks

In [ ]:
known_rule_names = sorted(set(rules["detail_clean"].dropna()) - {""})

def suggest_existing_rule(detail):
    if pd.isna(detail):
        return ""
    matches = get_close_matches(detail, known_rule_names, n=1, cutoff=0.78)
    return matches[0] if matches else ""

unmapped = (
    tidy.loc[tidy["classification_source"].eq("needs_review") & tidy["detail_clean"].notna()]
    .groupby(["category", "detail_clean"], dropna=False)
    .agg(count=("amount", "size"), total_amount=("amount", "sum"), first_date=("date", "min"), last_date=("date", "max"))
    .reset_index()
    .sort_values("total_amount", ascending=False)
)

if len(unmapped):
    unmapped["similar_existing_rule"] = unmapped["detail_clean"].apply(suggest_existing_rule)

summary = pd.DataFrame([{
    "rows": len(tidy),
    "total_amount": tidy["amount"].sum(),
    "start_date": tidy["date"].min(),
    "end_date": tidy["date"].max(),
    "missing_dates": tidy["date"].isna().sum(),
    "missing_amounts": tidy["amount"].isna().sum(),
    "negative_or_zero_amounts": (tidy["amount"] <= 0).sum(),
    "details_needing_review": len(unmapped),
}])

assert summary.loc[0, "missing_dates"] == 0, "Clean data has missing dates."
assert summary.loc[0, "missing_amounts"] == 0, "Clean data has missing amounts."
assert summary.loc[0, "negative_or_zero_amounts"] == 0, "Clean data has non-positive amounts."

summary_path = QUALITY_DIR / "cleaning_summary.csv"
unmapped_path = QUALITY_DIR / "unmapped_details.csv"
summary.to_csv(summary_path, index=False)
unmapped.to_csv(unmapped_path, index=False)

print(f"Saved summary: {summary_path}")
print(f"Saved unmapped details: {unmapped_path}")
display(summary)
display(unmapped.head(30))

## 11. Export the clean CSV for Tableau

In [ ]:
output_path = CLEAN_DIR / "personal_finance_transactions_clean.csv"
tidy.to_csv(output_path, index=False)
print(f"Saved clean CSV: {output_path}")
print(f"Rows: {len(tidy):,}")
print(f"Total amount: {tidy['amount'].sum():,.2f}")

## 12. Quick validation charts

These are not meant to replace Tableau. They are sanity checks: if the chart looks absurd, the cleaning or classification rules probably need attention.

In [ ]:
monthly = tidy.groupby("year_month", as_index=False)["amount"].sum()

plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly, x="year_month", y="amount", marker="o")
plt.xticks(rotation=45)
plt.title("Monthly spending")
plt.tight_layout()
plt.show()

In [ ]:
monthly_category = tidy.groupby(["year_month", "category"], as_index=False)["amount"].sum()

plt.figure(figsize=(14, 6))
sns.barplot(data=monthly_category, x="year_month", y="amount", hue="category")
plt.xticks(rotation=45)
plt.title("Monthly spending by original category")
plt.tight_layout()
plt.show()

In [ ]:
top_groups = tidy.groupby("detail_grouped", as_index=False)["amount"].sum().sort_values("amount", ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=top_groups, y="detail_grouped", x="amount")
plt.title("Top 15 spending groups")
plt.tight_layout()
plt.show()

In [ ]:
expense_monthly = tidy.groupby(["year_month", "expense_type"], as_index=False)["amount"].sum()

plt.figure(figsize=(12, 5))
sns.lineplot(data=expense_monthly, x="year_month", y="amount", hue="expense_type", marker="o")
plt.xticks(rotation=45)
plt.title("Operating vs capital/construction spending")
plt.tight_layout()
plt.show()

## 13. How to reduce future clerical work

When new unclassified names appear:

1. Open `data/quality/unmapped_details.csv`.
2. Add only the meaningful new names to `data/reference/classification_rules.csv`.
3. Rerun this notebook.

This is much faster than classifying everything inside notebook code. The notebook should be the process; the CSV should be the editable classification table.